In [ ]:
import os

# Set the visible devices to only GPU 2
# os.environ["CUDA_VISIBLE_DEVICES"] = "3"

# Verify CUDA device visibility in PyTorch
import torch
print("Available CUDA devices:", torch.cuda.device_count())
print("Using device:", torch.cuda.current_device())

import numpy as np
import DP as dp
import pandas as pd
import numpy as np
import process_edited as pce
from tqdm import tqdm
import itertools
import pickle

In [ ]:
hiv_dt = pd.read_csv("~/Documents/TimeAutoDiff/Dataset/hiv.csv.gz")
## train split
unique_id = hiv_dt['patient_id'].unique()
np.random.seed(123123)
unique_id = np.random.choice(unique_id, size=int(len(unique_id)*0.85), replace=False)

print(f'Training size: {len(unique_id)}')
test_df = hiv_dt.loc[~hiv_dt.patient_id.isin(unique_id)].reset_index(drop=True)
real_df = hiv_dt.loc[hiv_dt.patient_id.isin(unique_id)].reset_index(drop=True)

print(real_df.shape)

real_df1 = real_df.drop('date', axis=1)

In [ ]:
### For synteg

## Label:
real_lb = real_df.groupby('patient_id').first()[['enrol_d','age','male_y','center','mode']].reset_index()
real_lb.reset_index()
enrol_d = np.repeat(pd.cut(real_lb.enrol_d,bins=30,labels=False).values[:, np.newaxis],120,axis=1)
age = np.repeat(pd.cut(real_lb.age,bins=30,labels=False).values[:, np.newaxis],120,axis=1)
male_y = np.repeat(real_lb.male_y.astype('int').values[:, np.newaxis],120)
center = np.repeat(real_lb.center.astype('int').values[:, np.newaxis],120,axis=1)
center2site = {3: 2, 4: 2, 1: 1, 2: 2, 5: 3, 7: 5, 6: 5, 9: 6, 8: 4}
site = np.repeat(real_lb.center.map(center2site).astype('int').values[:, np.newaxis],120,axis=1)
mode = np.repeat(real_lb['mode'].astype('int').values[:, np.newaxis],120,axis=1)
np.save("./data/datasets/synteg/birth",enrol_d)
np.save("./data/datasets/synteg/age.npy",age)
np.save("./data/datasets/synteg/male_y.npy",male_y)
np.save("./data/datasets/synteg/center.npy",center)
np.save("./data/datasets/synteg/country.npy",site)
np.save("./data/datasets/synteg/mode.npy",mode)

print(f"enrol_d: {enrol_d.max()+1}")
print(f"age: {age.max()+1}")
print(f"male_y: {male_y.max()+1}")
print(f"center: {center.max()+1}")
print(f"center: {site.max()+1}")
print(f"mode: {mode.max()+1}")

## Visits:
real_df = real_df.drop(columns=['enrol_d','age','male_y','center','mode']).groupby('patient_id', group_keys=False).apply(lambda x: x.iloc[1:])

gap = pd.cut(real_df.pop('gap').fillna(0),bins=30,labels=False)
gap = pd.get_dummies(gap) * 1
gap["patient_id"] = real_df.patient_id.copy()
gap, _, _ = dp.partition_multi_seq(gap,1,'patient_id',120-1)
np.save("./data/datasets/synteg/interval.npy",gap.numpy())

## Continuous
cd4 = pd.get_dummies(pd.cut(real_df.pop('cd4_v'),bins=30,labels=False),prefix='cd4') * 1
rna = pd.get_dummies(pd.cut(real_df.pop('rna_v'),bins=30,labels=False),prefix='rna') * 1
weight = pd.get_dummies(pd.cut(real_df.pop('weight'),bins=30,labels=False),prefix='weight') * 1
height = pd.get_dummies(pd.cut(real_df.pop('height'),bins=30,labels=False),prefix='height') * 1


In [ ]:

real_df = real_df.drop(real_df.drop('date',axis=1).columns[real_df.drop(columns="date").std()==0],axis=1)
real_df = pd.concat([real_df,cd4,rna,weight,height],axis=1)

## Get Code
code,_,lengths = dp.partition_multi_seq(real_df,1,'patient_id',max_len=120-1)

lengths = 1 - lengths[...,-1].numpy()
lengths = lengths.sum(axis=1).astype(int) - 1
np.save("./data/datasets/synteg/length.npy",lengths)

# idx_sorted: (N, T, F), each slice gives feature-indices sorted by X-value
idx_sorted = torch.argsort(code, dim=2, descending=True)  # :contentReference[oaicite:1]{index=1}
D = int(code.sum(dim=2).max().item())
Y = idx_sorted[:, :, :D]  # shape (N, T, D)
mask = torch.arange(D)[None,None,:] < code.sum(dim=2, keepdim=True)
Y = torch.where(mask, Y, torch.tensor(-1, dtype=Y.dtype))
print(f"Max visit num: {Y.shape[-1]}")
print(f"code num: {Y.max() + 1}")

# ## Add EOS
# Z = Y[np.arange(Y.shape[0]),lengths.astype(int)]
# Z = torch.concat([Y.max() * torch.ones(Y.shape[0]).unsqueeze(-1) + 1,
#                   Z[:,:-1]],dim=1)
# Y[np.arange(Y.shape[0]),lengths.astype(int)] = Z.long()
np.save("./data/datasets/synteg/code.npy",Y.numpy())